# Generador de Scripts SQL para Base de Datos Athena

Este notebook genera los scripts SQL para insertar datos en la base de datos a partir del archivo Excel.

In [1]:
import pandas as pd
from pathlib import Path

In [11]:
class Extractor:
    def __init__(self, ruta_excel: str, ruta_sql_output: str):
        self.ruta_excel = Path(ruta_excel)
        self.ruta_sql_output = Path(ruta_sql_output)
        self.data = pd.read_excel(self.ruta_excel, sheet_name=None)
        
        print(f"📊 Excel cargado: {self.ruta_excel}")
        print(f"📁 Output SQL: {self.ruta_sql_output}")
        print(f"📋 Hojas disponibles: {list(self.data.keys())}")
    
    def extract(self) -> str:
        all_dml = []
        for sheet_name in self.data.keys():
            match sheet_name.lower():
                case "area":
                    dml_area = self._extract_areas()
                    all_dml.append(dml_area)
                
                case "escuelas":
                    dml_escuelas = self._extract_escuelas()
                    all_dml.append(dml_escuelas)
                
                case "carreras":
                    dml_carreras = self._extract_carreras()
                    all_dml.append(dml_carreras)
                
                case "carrera_escuelas":
                    dml_escuela_carrera = self._extract_escuela_carrera()
                    all_dml.append(dml_escuela_carrera)
                
                case _:
                    raise ValueError(f"Hoja '{sheet_name}' no tiene extractor implementado")

        self._save_sql(all_dml)
    
    def _extract_areas(self) -> str:
        df = self.data['area']
        sql_lines = [
            "-- ============================================",
            "-- INSERCIÓN DE ÁREAS ACADÉMICAS",
            "-- ============================================",
            "",
            "BEGIN;",
            "",
            "INSERT INTO public.areas (area_id, nom_area) VALUES"
        ]
        
        values = []
        for _, row in df.iterrows():
            area_id = row['id_area']
            nombre_area = row['nombre_area'].replace("'", "''")
            values.append(f"    ({area_id}, '{nombre_area}')")
        
        sql_lines.append(",\n".join(values) + ";")
        sql_lines.extend([
            "",
            "COMMIT;",
        ])
        
        return "\n".join(sql_lines)
    
    def _extract_escuelas(self) -> str:
        df = self.data['escuelas']
        sql_lines = [
            "-- ============================================",
            "-- INSERCIÓN DE ESCUELAS",
            "-- ============================================",
            "",
            "BEGIN;",
            "",
            "INSERT INTO public.escuelas (escuela_id, nom_escuela) VALUES"
        ]
        
        values = []
        for _, row in df.iterrows():
            escuela_id = row['id_escuela']
            nombre_escuela = row['nombre_escuela'].replace("'", "''")
            values.append(f"    ({escuela_id}, '{nombre_escuela}')")
        
        sql_lines.append(",\n".join(values) + ";")
        sql_lines.extend([
            "",
            "COMMIT;",
        ])
        
        return "\n".join(sql_lines)
    
    def _extract_carreras(self) -> str:
        df = self.data['carreras']
        sql_lines = [
            "-- ============================================",
            "-- INSERCIÓN DE CARRERAS",
            "-- ============================================",
            "",
            "BEGIN;",
            "",
            "INSERT INTO public.carreras (carrera_id, area_id, nom_carrera, origina_objetivo, objetivo, pregunta_objetivo) VALUES"
        ]
        
        values = []
        for _, row in df.iterrows():
            carrera_id = row['id_carrera']
            area_id = row['id_area']
            nom_carrera = row['nombre_carrera'].replace("'", "''")
            objetivo_resumido = row['objetivo_resumido'].replace("'", "''")
            pregunta_objetivo = row['pregunta'].replace("'", "''")
            origina_objetivo = row['objetivo'].replace("'", "''")
            
            values.append(
                f"    ({carrera_id}, {area_id}, '{nom_carrera}', '{origina_objetivo}', '{objetivo_resumido}', '{pregunta_objetivo}')"
            )
        
        sql_lines.append(",\n".join(values) + ";")
        sql_lines.extend([
            "",
            "COMMIT;",
        ])
        
        return "\n".join(sql_lines)
    
    def _extract_escuela_carrera(self) -> str:
        df = self.data['carrera_escuelas']
        sql_lines = [
            "-- ============================================",
            "-- INSERCIÓN DE RELACIÓN ESCUELA-CARRERA",
            "-- ============================================",
            "",
            "BEGIN;",
            "",
            "INSERT INTO public.escuela_carrera (id_escuela, id_carrera) VALUES"
        ]
        
        values = []
        for _, row in df.iterrows():
            id_escuela = row['id_escuela']
            id_carrera = row['id_carrera']
            values.append(f"    ({id_escuela}, {id_carrera})")
        
        sql_lines.append(",\n".join(values) + ";")
        sql_lines.extend([
            "",
            "COMMIT;",
        ])
        
        return "\n".join(sql_lines)
    
    def _save_sql(self, sql_content_list: list):
        output_file = self.ruta_sql_output / f"insert_data.sql"

        # Encabezado del archivo
        header = [
            "-- ============================================",
            "-- SCRIPT DE INSERCIÓN DE DATOS - ATHENA",
            "-- Generado automáticamente",
            f"-- Archivo: {self.ruta_excel.name}",
            "-- ============================================",
            ""
        ]

        # Unir todos los DML con separadores
        full_sql = "\n".join(header) + "\n\n" + "\n\n".join(sql_content_list)

        # Escribir todo de una vez
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(full_sql)

        print(f"✅ SQL guardado: {output_file}")

In [12]:
RUTA_EXCEL = "../assets/layouts/excel/tablas_completas.xlsx"
RUTA_SQL_OUTPUT = "/Users/jmonroy/Documents/MyProjects/Athena/DataBase/scripts"
if __name__ == "__main__":
    Path(RUTA_SQL_OUTPUT).mkdir(parents=True, exist_ok=True)
    extractor = Extractor(RUTA_EXCEL, RUTA_SQL_OUTPUT)
    extractor.extract()

📊 Excel cargado: ../assets/layouts/excel/tablas_completas.xlsx
📁 Output SQL: /Users/jmonroy/Documents/MyProjects/Athena/DataBase/scripts
📋 Hojas disponibles: ['area', 'carreras', 'escuelas', 'carrera_escuelas']
✅ SQL guardado: /Users/jmonroy/Documents/MyProjects/Athena/DataBase/scripts/insert_data.sql
